Aqui vamos a desormalizar las columnas del parquet entrenado por el GAN.

1. Product Cateogory
2. Zonas (polygon id)

In [1]:
# --- 1. CONFIGURACIÓN DE ENTORNO Y LIBRERÍAS ---
import os

# Instalación de Java 11 (Requerido para Spark y BigQuery Connector)
print("Instalando Java 11...")
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Instalación de PySpark y FindSpark
print("Instalando PySpark 3.4.1...")
!pip install -q pyspark==3.4.1 findspark

print("--- Entorno Configurado ---")
!java -version

Instalando Java 11...
Instalando PySpark 3.4.1...
--- Entorno Configurado ---
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [2]:
# --- 2. AUTENTICACIÓN ---
from google.colab import auth

try:
    auth.authenticate_user()
    print('Autenticado exitosamente en Google Cloud')
except:
    print('Ya estabas autenticado o hubo un error manual.')

Autenticado exitosamente en Google Cloud


In [3]:
# --- 3. INICIALIZACIÓN DE SPARK SESSION ---
import findspark
from pyspark.sql import SparkSession

findspark.init()

# Inicializamos Spark incluyendo el conector de BigQuery
spark = SparkSession.builder \
    .appName("Pienza_Denormalization") \
    .config("spark.jars.packages", "com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.34.0") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print(f"--- Spark Iniciado ---")
print(f"Versión: {spark.version}")

--- Spark Iniciado ---
Versión: 3.4.1


In [4]:
# --- 4. CARGA DE DATOS DESDE DRIVE (DATASET MAESTRO) ---
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Ruta exacta al dataset completo (Output del GAN)
PARQUET_SOURCE_PATH = "/content/drive/MyDrive/_Pienza/Assets/Phase_4/pienza_manifold_v3.parquet"

print(f"Cargando dataset maestro desde: {PARQUET_SOURCE_PATH} ...")

# 3. Leer el archivo Parquet con Spark
# Spark lee los metadatos y el esquema automáticamente
try:
    df_manifold = spark.read.parquet(PARQUET_SOURCE_PATH)

    # 4. Registrar como vista temporal para SQL
    # Esto nos permitirá hacer: SELECT DISTINCT ... para crear los diccionarios
    df_manifold.createOrReplaceTempView("simulacion_manifold")

    print("--- ¡ÉXITO! DATASET CARGADO ---")
    print(f"Total de filas: {df_manifold.count()}")
    df_manifold.printSchema()

except Exception as e:
    print(f"ERROR: No se pudo leer el archivo en: {PARQUET_SOURCE_PATH}")
    print(e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando dataset maestro desde: /content/drive/MyDrive/_Pienza/Assets/Phase_4/pienza_manifold_v3.parquet ...
--- ¡ÉXITO! DATASET CARGADO ---
Total de filas: 1010001
root
 |-- upfront_fare: float (nullable = true)
 |-- est_trip_time_sec: float (nullable = true)
 |-- est_trip_dist_km: float (nullable = true)
 |-- time_to_pickup_sec: float (nullable = true)
 |-- dist_to_pickup_km: float (nullable = true)
 |-- hour_of_day: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- product_category_fk: string (nullable = true)
 |-- dropoff_zone_id: string (nullable = true)
 |-- pickup_h3_id: string (nullable = true)
 |-- reason_primary_fk: string (nullable = true)
 |-- eph_operational: float (nullable = true)
 |-- vexation_index: float (nullable = true)



In [5]:
# --- 5. CONFIGURACIÓN DE DESTINO BIGQUERY ---

# Tus variables de proyecto
GCP_PROJECT_ID = "645009831643"
BQ_DATASET     = "pienza_synth"

# Configuración temporal para escrituras en BQ (necesario para Spark)
spark.conf.set("temporaryGcsBucket", "pienza_synth_temp_bucket") # Ajusta si tienes un bucket específico, o Spark intentará crear uno

print(f"Configurado para escribir en: {GCP_PROJECT_ID}.{BQ_DATASET}")

Configurado para escribir en: 645009831643.pienza_synth


In [6]:
# ==============================================================================
# CELL 6: THE UNIVERSAL LOOKUP TABLE (BULLETPROOF VERSION)
# ==============================================================================
from pyspark.sql import functions as F

print("🏗️  Forjando la Piedra Rosetta de Pienza (Protocolo de Máxima Robustez)...")

# --- 1. CONFIGURACIÓN DE IDENTIDAD (Asegurar ID de Texto) ---
# Usamos el ID de texto, que es el estándar para recursos en GCP
BILLING_PROJECT_ID = "drivers-dilemma"
DATASET_REALIDAD = "pienza_mini"

# --- 2. DICCIONARIO HUMANO (P_ Axis) ---
salchichota_data = [
    (0, "santa_fe_bosques_de__santa_fe_cumbres_de__santa_fe_tec"),
    (1, "santa_fe_centro_comercial"),
    (2, "carretera_al_olivo__carretera_libre__cruce_echanove__vistahermosa"),
    (3, "bosques_pabellon__el_olivo__loma_de_la_palma"),
    (4, "agwa_bezares__reforma_bnp"), (5, "ahuehuetes_norte__de_los_bosques"),
    (6, "interlomas_haciendas__jesus_del_monte"), (7, "blvrd_anahuac__universidad_anahuac"),
    (8, "ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca"),
    (9, "santa_fe_ibero"), (10, "lomas_altas__nodo_reforma_palmas__reforma_regina"),
    (11, "ahuehuetes_sur"), (12, "tamarindos"), (13, "santa_fe_quintana__sante_fe_patio"),
    (14, "bosque_real__lomas_country_club"), (15, "herradura_conscripto"),
    (16, "de_las_fuentes__tecamachalco"), (17, "lomas_barrilaco__lomas_olimpo__nodo_monte_libano"),
    (18, "lomas_prado_norte__lomas_trastevere"), (19, "lomas_virreyes"),
    (20, "lomas_fc_cuernavaca"), (21, "fuentes_casino__sedena__tecamachalco"),
    (22, "palmas_jp_morgan"), (23, "bondojito_asf__bosque_2__bosque_3"),
    (24, "bosque_1__campo_marte"), (25, "roma_condesa_2"), (26, "rios"),
    (27, "roma_condesa_1"), (28, "juarez_soho_house"), (29, "anzures"),
    (30, "anahuac_1__bahias__frontera_polanco"), (31, "sotelo"),
    (32, "carso_antara_miyana"), (33, "irrigacion__polanco_uber_hq"),
    (34, "juarez_rosa"), (35, "lagos"), (36, "polanco_grupo_mexico__polanco_palacio"),
    (37, "polanco_gandhi"), (38, "polanco_5"), (39, "polanco_parque_lincoln"),
    (40, "polanco_parroquia"), (41, "santa_fe_colegios")
]
df_dict_p = spark.createDataFrame(salchichota_data, ["id_raw", "semantic_name"]) \
                 .withColumn("zone_key", F.concat(F.lit("P_"), F.col("id_raw").cast("string"))) \
                 .select("zone_key", "semantic_name")

# --- 3. DICCIONARIO MÁQUINA (C_ Axis) ---
try:
    print(f"📡 Recuperando nombres de clusters desde {DATASET_REALIDAD}.silver_palette...")
    # Agregamos parentProject para que Google sepa a quién cobrar los bytes escaneados
    df_silver_palette = spark.read.format("bigquery") \
        .option("table", f"{BILLING_PROJECT_ID}.{DATASET_REALIDAD}.silver_palette") \
        .option("parentProject", BILLING_PROJECT_ID) \
        .load()

    df_dict_c = df_silver_palette \
        .select("dropoff_hdbscan_id", "dropoff_hdbscan_name") \
        .distinct() \
        .filter(F.col("dropoff_hdbscan_id") != -1) \
        .withColumn("zone_key", F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string"))) \
        .select("zone_key", F.col("dropoff_hdbscan_name").alias("semantic_name"))

    print(f"   ✅ Metadatos de máquina recuperados: {df_dict_c.count()} clústeres.")

except Exception as e:
    print("🔴 FALLO CRÍTICO AL LEER BIGQUERY.")
    print("Posibles causas: El nombre 'drivers-dilemma' es incorrecto o falta el parentProject.")
    raise e

# --- 4. UNIFICACIÓN FINAL ---
df_unassigned = spark.createDataFrame([("Unassigned", "unassigned_area")], ["zone_key", "semantic_name"])

# Unión Triple
df_master_dictionary = df_dict_p.union(df_dict_c).union(df_unassigned)

print(f"✅ Diccionario maestro unificado: {df_master_dictionary.count()} zonas.")
df_master_dictionary.orderBy("zone_key").show(10, truncate=False)

🏗️  Forjando la Piedra Rosetta de Pienza (Protocolo de Máxima Robustez)...
📡 Recuperando nombres de clusters desde pienza_mini.silver_palette...
   ✅ Metadatos de máquina recuperados: 46 clústeres.
✅ Diccionario maestro unificado: 89 zonas.
+--------+-----------------------+
|zone_key|semantic_name          |
+--------+-----------------------+
|C_-2    |missing_coordinates    |
|C_0     |viaducto_tlalpan       |
|C_1     |terminal_1_aicm        |
|C_10    |observatorio           |
|C_11    |sotelo_san_esteban     |
|C_12    |barranca_del_muerto    |
|C_13    |haciendas_san_fernando |
|C_14    |vialidad_de_la_barranca|
|C_15    |interlomas_magnocentro |
|C_16    |santa_fe_itesm         |
+--------+-----------------------+
only showing top 10 rows



In [7]:
# ==============================================================================
# CELL 6.1: FULL GEOGRAPHIC RECONNAISSANCE (THE 89 ZONES)
# ==============================================================================
# Purpose: Display the complete Pienza dictionary for manual audit.
# ==============================================================================

print("📖 LISTADO MAESTRO DE ZONAS (89 IDENTIDADES SEMÁNTICAS)")
print("-" * 80)

# Convertimos a Pandas para visualización premium en Colab
df_full_audit = df_master_dictionary.orderBy("zone_key").toPandas()

# Configuramos Pandas para que no trunque los nombres largos de la "Salchichota"
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

display(df_full_audit)

print("-" * 80)
print(f"✅ Total: {len(df_full_audit)} registros listos para el JOIN del millón.")

📖 LISTADO MAESTRO DE ZONAS (89 IDENTIDADES SEMÁNTICAS)
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_0,viaducto_tlalpan
2,C_1,terminal_1_aicm
3,C_10,observatorio
4,C_11,sotelo_san_esteban
5,C_12,barranca_del_muerto
6,C_13,haciendas_san_fernando
7,C_14,vialidad_de_la_barranca
8,C_15,interlomas_magnocentro
9,C_16,santa_fe_itesm


--------------------------------------------------------------------------------
✅ Total: 89 registros listos para el JOIN del millón.


In [8]:
# ==============================================================================
# CELL 6.2: THE GEOGRAPHIC PURGE (NON-DESTRUCTIVE)
# ==============================================================================

print("🧹 Identificando zonas activas en el Manifold...")

# 1. Guardamos una copia del diccionario completo (89 filas) antes de filtrar
df_master_dictionary_full = df_master_dictionary

# 2. Identificar los IDs que REALMENTE existen en el Manifold (1M de filas)
df_active_keys = df_manifold.select("dropoff_zone_id").distinct()

# 3. Crear el diccionario curado (67 filas)
df_dict_curated = df_master_dictionary_full.join(
    df_active_keys,
    df_master_dictionary_full.zone_key == df_active_keys.dropoff_zone_id,
    "inner"
).select("zone_key", "semantic_name")

print(f"   ✅ Filtrado completado: 89 -> {df_dict_curated.count()} zonas vivas.")

# Sobrescribimos el puntero principal para el JOIN final
df_master_dictionary = df_dict_curated

🧹 Identificando zonas activas en el Manifold...
   ✅ Filtrado completado: 89 -> 67 zonas vivas.


In [9]:
# ==============================================================================
# CELL 6.3: THE SHADOW AUDIT (IDENTIFYING DELETED ZONES)
# ==============================================================================

print("🕵️‍♂️ Investigando las 22 zonas eliminadas...")

# Buscamos lo que está en el FULL pero NO en el CURATED
df_deleted_zones = df_master_dictionary_full.join(
    df_active_keys,
    df_master_dictionary_full.zone_key == df_active_keys.dropoff_zone_id,
    "left_anti"
)

# Convertimos a Pandas para el reporte
df_deleted_report = df_deleted_zones.orderBy("zone_key").toPandas()

print(f"\n🚫 LISTADO DE EXCLUSIÓN (Zonas Sombra / Absorbidas):")
print("-" * 80)
if len(df_deleted_report) > 0:
    display(df_deleted_report)
else:
    print("✨ No se encontraron zonas eliminadas.")

print("-" * 80)
print(f"✅ Auditoría terminada. Estas zonas ya no contaminarán el Manifold.")

🕵️‍♂️ Investigando las 22 zonas eliminadas...

🚫 LISTADO DE EXCLUSIÓN (Zonas Sombra / Absorbidas):
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_14,vialidad_de_la_barranca
2,C_20,tamarindos
3,C_21,nodo_constituyentes_reforma
4,C_24,santa_fe_core
5,C_25,pabellon_bosques
6,C_26,duraznos
7,C_27,chamizal
8,C_28,lomas_anahuac
9,C_31,san_miguel_chapultepec


--------------------------------------------------------------------------------
✅ Auditoría terminada. Estas zonas ya no contaminarán el Manifold.


In [10]:
# ==============================================================================
# CELL 6.4: THE GHOST RECOVERY (FORCED COMPARISON)
# ==============================================================================

print("🕵️‍♂️  Recuperando los 22 fantasmas desde los cimientos...")

# 1. Reconstruimos el diccionario completo (P_ + C_)
df_reconstructed_full = df_dict_p.union(df_dict_c).union(df_unassigned)

# 2. Identificamos los IDs que realmente están en el Manifold (1M)
df_active_manifold_keys = df_manifold.select("dropoff_zone_id").distinct()

# 3. ANTI-JOIN: ¿Qué estaba en el diccionario pero NO llegó al Manifold?
df_ghosts = df_reconstructed_full.join(
    df_active_manifold_keys,
    df_reconstructed_full.zone_key == df_active_manifold_keys.dropoff_zone_id,
    "left_anti"
)

# 4. Reporte Final
ghosts_pd = df_ghosts.orderBy("zone_key").toPandas()

print(f"\n🚫  ZONAS PURGADAS ({len(ghosts_pd)} registros):")
print("-" * 80)
if len(ghosts_pd) > 0:
    display(ghosts_pd)
    print("\n💡 ANÁLISIS SOBERANO: Estas zonas fueron absorbidas por tus polígonos P_")
    print("   o el GAN determinó que su probabilidad de ocurrencia es cero.")
else:
    print("🔎 Algo sigue mal en la referencia. Procedamos al JOIN para no perder el momentum.")

# Aseguramos que para el JOIN final usamos la versión curada
df_master_dictionary = df_reconstructed_full.join(
    df_active_manifold_keys,
    df_reconstructed_full.zone_key == df_active_manifold_keys.dropoff_zone_id,
    "inner"
).select("zone_key", "semantic_name")

🕵️‍♂️  Recuperando los 22 fantasmas desde los cimientos...

🚫  ZONAS PURGADAS (22 registros):
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_14,vialidad_de_la_barranca
2,C_20,tamarindos
3,C_21,nodo_constituyentes_reforma
4,C_24,santa_fe_core
5,C_25,pabellon_bosques
6,C_26,duraznos
7,C_27,chamizal
8,C_28,lomas_anahuac
9,C_31,san_miguel_chapultepec



💡 ANÁLISIS SOBERANO: Estas zonas fueron absorbidas por tus polígonos P_
   o el GAN determinó que su probabilidad de ocurrencia es cero.


In [11]:
# ==============================================================================
# CELL 6.5: REAL-WORLD COMPOSITION AUDIT (THE 67 SURVIVORS - BUG FIX)
# ==============================================================================
from pyspark.sql import functions as F
from itertools import chain

print("📡 Iniciando Auditoría Forense de la Realidad...")

# --- 1. RE-INYECCIÓN DEL MAPA CANÓNICO ---
id_map = {
    -1: -1, 41: 0, 42: 0, 46: 0, 43: 1, 65: 2, 62: 2, 44: 2, 36: 2, 49: 3, 52: 3,
    35: 3, 50: 4, 58: 4, 25: 5, 31: 5, 63: 6, 39: 6, 51: 7, 33: 7, 37: 8, 53: 8,
    48: 8, 60: 9, 57: 10, 12: 10, 32: 10, 24: 11, 40: 12, 45: 13, 59: 13, 61: 14,
    38: 14, 34: 15, 30: 16, 66: 16, 17: 17, 14: 17, 22: 17, 16: 18, 13: 18, 11: 19,
    15: 20, 21: 21, 20: 21, 19: 21, 18: 22, 47: 23, 55: 23, 56: 23, 54: 24, 64: 24,
    71: 25, 9: 26, 70: 27, 69: 28, 8: 29, 6: 30, 7: 30, 23: 30, 3: 31, 2: 32,
    4: 33, 29: 33, 68: 34, 5: 35, 27: 36, 28: 36, 1: 37, 10: 38, 0: 39, 26: 40, 67: 41
}

try:
    # --- 2. CARGA DESDE BIGQUERY ---
    df_real_raw = spark.read.format("bigquery") \
        .option("parentProject", BILLING_PROJECT_ID) \
        .option("table", f"{BILLING_PROJECT_ID}.{DATASET_REALIDAD}.v_ML_Supervised") \
        .option("viewsEnabled", "true") \
        .option("materializationDataset", DATASET_REALIDAD) \
        .load() \
        .select("offer_id", "dropoff_polygon_id", "dropoff_hdbscan_id")

    # --- 3. PROCESAMIENTO SPATIAL ---
    # Convertimos id_map a una columna tipo Map de Spark
    mapping_expr = F.create_map([F.lit(x) for x in chain(*id_map.items())])

    # Aplicamos Coalesce y Mapeo (Sintaxis Spark 3.0+)
    df_real_mapped = df_real_raw.withColumn(
        "id_agrupado",
        F.coalesce(mapping_expr[F.col("dropoff_polygon_id").cast("int")], F.lit(-1))
    )

    # Construimos la llave zone_key
    df_real_final = df_real_mapped.withColumn("zone_key",
        F.when(F.col("id_agrupado") >= 0, F.concat(F.lit("P_"), F.col("id_agrupado").cast("string")))
         .when(F.col("dropoff_hdbscan_id") > -1, F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string")))
         .otherwise("Unassigned")
    )

    # Conteo de frecuencia real
    df_real_counts = df_real_final.groupBy("zone_key").count().withColumnRenamed("count", "n_obs_real")

    # --- 4. JOIN Y LIMPIEZA DE NULOS (FIXED) ---
    # Usamos F.coalesce para llenar los nulos en la columna resultante del join
    df_final_audit_table = df_master_dictionary.join(
        df_real_counts,
        "zone_key",
        "left"
    ).select(
        "zone_key",
        "semantic_name",
        F.coalesce(F.col("n_obs_real"), F.lit(0)).alias("n_obs_real")
    ).orderBy(F.col("n_obs_real").desc())

    # --- 5. REPORTE FINAL ---
    print("\n🏆 CENSO REALIDAD (67 ZONAS VIVAS):")
    final_report_pd = df_final_audit_table.toPandas()
    display(final_report_pd)

    print("-" * 80)
    print(f"✅ Auditoría completada. El ADN real está sincronizado.")

except Exception as e:
    print(f"🔴 ERROR EN LA AUDITORÍA: {e}")
    raise e

📡 Iniciando Auditoría Forense de la Realidad...

🏆 CENSO REALIDAD (67 ZONAS VIVAS):


,zone_key,semantic_name,n_obs_real
0,Unassigned,unassigned_area,1366
1,P_25,roma_condesa_2,223
2,P_1,santa_fe_centro_comercial,169
3,P_9,santa_fe_ibero,150
4,P_32,carso_antara_miyana,122
5,C_1,terminal_1_aicm,111
6,P_13,santa_fe_quintana__sante_fe_patio,104
7,P_15,herradura_conscripto,99
8,P_8,ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca,96
9,P_12,tamarindos,82


--------------------------------------------------------------------------------
✅ Auditoría completada. El ADN real está sincronizado.


### 📝 Nota Técnica: Auditoría de Residuales Topológicos (P_ vs. C_)

Tras ejecutar el censo de densidad en la realidad (**Cell 6.5**), se identificaron 6 zonas de origen algorítmico (**C_**) con un conteo de observaciones inusualmente bajo ($N < 25$), destacando casos como `C_18 (santa_fe_patio)` y `C_32 (tacubaya)` con $N=1$.

**Análisis de la Anomalía:**
Aunque el objetivo inicial del *Master Coalesce* era que los polígonos humanos absorbierean la mayor parte de la actividad, estos residuales representan un fenómeno de **"Spillover Geo-Estadístico"**:

1.  **Límites Duros vs. Densidad Suave:** Los polígonos manuales tienen fronteras geométricas rígidas. HDBSCAN, por el contrario, agrupa por densidad. Estos puntos son ofertas que el algoritmo detectó como parte del "clúster" (ej. el ecosistema del ITESM), pero que físicamente cayeron a unos metros de distancia, fuera del trazo manual del polígono.
2.  **Artefactos Naturales:** Casos como `santa_fe_itesm` ($N=6$) o `santa_fe_abc` ($N=8$) son críticos. Representan la "periferia" de zonas de alta demanda que la máquina reconoce aunque el ojo humano no las haya encerrado.

**Decisión Arquitectónica:**
Se ha decidido **PRESERVAR** estas 67 zonas (incluyendo los residuales) en el Manifold por dos razones:
*   **Fidelidad del Generador:** El GAN v2 aprendió la existencia de estos micro-puntos. Eliminarlos ahora crearía una inconsistencia entre el modelo entrenado y el diccionario de salida.
*   **Rigor Forense:** Estos puntos representan la "frontera" real del mercado. En la simulación de la Fase VI, estas zonas actuarán como *edge cases* naturales, permitiendo testear la sensibilidad del modelo en zonas de baja visibilidad.

**Visto Bueno (VoBo):** La topología se considera finalizada y validada. Procediendo a la denormalización masiva del millón de registros.

In [12]:
# ==============================================================================
# CELL 6.7: THE UNIFIED DICTIONARY FORGE & PERSISTENCE (v2.0)
# ==============================================================================
# Purpose: Manually forge and persist both Product and Geographic dictionaries.
# Assets:  dim_product_hierarchy.parquet & dim_zone_dictionary.parquet
# ==============================================================================
from pyspark.sql import functions as F

# --- 1. CONFIGURACIÓN DE RUTAS ---
PROD_DICT_PATH = "/content/drive/MyDrive/_Pienza/Assets/Phase_6/dim_product_hierarchy.parquet"
ZONE_DICT_PATH = "/content/drive/MyDrive/_Pienza/Assets/Phase_6/dim_zone_dictionary.parquet"

print("🏗️  FORJANDO BÓVEDA DE METADATOS PIENZA...")
print("-" * 65)

# --- 2. DICCIONARIO DE PRODUCTOS (JERARQUÍA 1-2-3) ---
products_dna = [(1, "X"), (2, "Mid-Tier"), (3, "Premium")]
df_dim_products = spark.createDataFrame(products_dna, ["id_p", "product_name"])

# --- 3. DICCIONARIO DE GEOGRAFÍA (SALCHICHOTA + SURVIVORS) ---
# A. Nombres Manuales (P_ Axis)
salchichota_data = [
    (0, "santa_fe_bosques_de__santa_fe_cumbres_de__santa_fe_tec"),
    (1, "santa_fe_centro_comercial"),
    (2, "carretera_al_olivo__carretera_libre__cruce_echanove__vistahermosa"),
    (3, "bosques_pabellon__el_olivo__loma_de_la_palma"),
    (4, "agwa_bezares__reforma_bnp"), (5, "ahuehuetes_norte__de_los_bosques"),
    (6, "interlomas_haciendas__jesus_del_monte"), (7, "blvrd_anahuac__universidad_anahuac"),
    (8, "ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca"),
    (9, "santa_fe_ibero"), (10, "lomas_altas__nodo_reforma_palmas__reforma_regina"),
    (11, "ahuehuetes_sur"), (12, "tamarindos"), (13, "santa_fe_quintana__sante_fe_patio"),
    (14, "bosque_real__lomas_country_club"), (15, "herradura_conscripto"),
    (16, "de_las_fuentes__tecamachalco"), (17, "lomas_barrilaco__lomas_olimpo__nodo_monte_libano"),
    (18, "lomas_prado_norte__lomas_trastevere"), (19, "lomas_virreyes"),
    (20, "lomas_fc_cuernavaca"), (21, "fuentes_casino__sedena__tecamachalco"),
    (22, "palmas_jp_morgan"), (23, "bondojito_asf__bosque_2__bosque_3"),
    (24, "bosque_1__campo_marte"), (25, "roma_condesa_2"), (26, "rios"),
    (27, "roma_condesa_1"), (28, "juarez_soho_house"), (29, "anzures"),
    (30, "anahuac_1__bahias__frontera_polanco"), (31, "sotelo"),
    (32, "carso_antara_miyana"), (33, "irrigacion__polanco_uber_hq"),
    (34, "juarez_rosa"), (35, "lagos"), (36, "polanco_grupo_mexico__polanco_palacio"),
    (37, "polanco_gandhi"), (38, "polanco_5"), (39, "polanco_parque_lincoln"),
    (40, "polanco_parroquia"), (41, "santa_fe_colegios")
]
df_p = spark.createDataFrame(salchichota_data, ["id", "semantic_name"]) \
            .withColumn("zone_key", F.concat(F.lit("P_"), F.col("id").cast("string"))) \
            .select("zone_key", "semantic_name")

# B. Nombres Máquina (C_ Axis desde BigQuery)
print("📡 Recuperando nombres de clusters desde BQ...")
df_c = spark.read.format("bigquery") \
        .option("table", f"{BILLING_PROJECT_ID}.pienza_mini.silver_palette") \
        .option("parentProject", BILLING_PROJECT_ID) \
        .load() \
        .select("dropoff_hdbscan_id", "dropoff_hdbscan_name") \
        .distinct() \
        .filter(F.col("dropoff_hdbscan_id") != -1) \
        .withColumn("zone_key", F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string"))) \
        .select("zone_key", F.col("dropoff_hdbscan_name").alias("semantic_name"))

# C. Unassigned & Unión
df_u = spark.createDataFrame([("Unassigned", "unassigned_area")], ["zone_key", "semantic_name"])
df_full_dict = df_p.union(df_c).union(df_u)

# D. Filtro de Supervivencia (Solo las 67 que existen en el Manifold)
print("🧹 Filtrando por supervivencia (Las 67 vivas)...")
df_active_keys = df_manifold.select("dropoff_zone_id").distinct()
df_dim_zones = df_full_dict.join(df_active_keys, df_full_dict.zone_key == df_active_keys.dropoff_zone_id, "inner") \
                           .select("zone_key", "semantic_name")

# --- 4. PERSISTENCIA FÍSICA ---
try:
    print(f"💾 Persistiendo en Drive...")
    df_dim_products.toPandas().to_parquet(PROD_DICT_PATH, index=False)
    df_dim_zones.toPandas().to_parquet(ZONE_DICT_PATH, index=False)

    print("\n✅ BÓVEDA ACTUALIZADA Y ASEGURADA.")
    print(f"   - Productos: {df_dim_products.count()} niveles (X, Mid, Premium)")
    print(f"   - Geografía: {df_dim_zones.count()} zonas (Salchichota + Clusters)")
except Exception as e:
    print(f"❌ Error en la persistencia: {e}")

print("-" * 65)

🏗️  FORJANDO BÓVEDA DE METADATOS PIENZA...
-----------------------------------------------------------------
📡 Recuperando nombres de clusters desde BQ...
🧹 Filtrando por supervivencia (Las 67 vivas)...
💾 Persistiendo en Drive...

✅ BÓVEDA ACTUALIZADA Y ASEGURADA.
   - Productos: 3 niveles (X, Mid, Premium)
   - Geografía: 67 zonas (Salchichota + Clusters)
-----------------------------------------------------------------


In [13]:
# ==============================================================================
# CELL 7: THE STAR SCHEMA VALIDATION (DENORMALIZING THE MILLION - FIXED)
# ==============================================================================
# Purpose: Final union of Atoms and Dimensions using explicit join conditions.
# Strategy: Broadcast Join (Map small dictionaries across the Spark cluster).
# ==============================================================================
from pyspark.sql import functions as F

print("🚀 Inyectando Inteligencia Semántica al Manifold (1,010,001 filas)...")

# 1. CARGA DE ACTIVOS DESDE DRIVE (Sincronización con el disco)
df_facts = spark.read.parquet(PARQUET_SOURCE_PATH)
df_dim_zones = spark.read.parquet(ZONE_DICT_PATH)
df_dim_prods = spark.read.parquet(PROD_DICT_PATH)

# 2. JOIN 1: JERARQUÍA DE PRODUCTO (X, Mid-Tier, Premium)
# Unimos product_category_fk con id_p
df_named_step1 = df_facts.join(
    F.broadcast(df_dim_prods),
    df_facts.product_category_fk == df_dim_prods.id_p,
    "left"
).drop("id_p")

# 3. JOIN 2: TOPOLOGÍA ESTRATÉGICA (Salchichota + Clusters)
# FIX: Unimos dropoff_zone_id (Hechos) con zone_key (Diccionario)
df_final = df_named_step1.join(
    F.broadcast(df_dim_zones),
    df_named_step1.dropoff_zone_id == df_dim_zones.zone_key,
    "left"
).drop("zone_key") # Eliminamos la llave duplicada para limpieza

# 4. AUDITORÍA DE INTEGRIDAD
# Verificamos si hubo "huérfanos" (zonas que no encontraron su nombre semántico)
orphans = df_final.filter(F.col("semantic_name").isNull()).count()

if orphans == 0:
    print("✅ VOBO TOTAL: Sincronización perfecta. 100% de las zonas tienen nombre.")
else:
    print(f"⚠️  ALERTA: Se detectaron {orphans} registros sin nombre asignado. Revisar diccionarios.")

# 5. MUESTRA DE VALIDACIÓN (Híbrido P_ y C_)
print("\n🧪 MUESTRA ALEATORIA DEL UNIVERSO DENORMALIZADO:")
print("-" * 80)
# Mostramos registros que no sean Unassigned para validar la Salchichota
df_final.select(
    "product_name",
    "dropoff_zone_id",
    "semantic_name",
    "upfront_fare",
    "est_trip_time_sec"
).filter(F.col("dropoff_zone_id") != "Unassigned") \
 .orderBy(F.rand()) \
 .show(20, truncate=False)

# 6. PERSISTENCIA EN CACHÉ
df_final.cache()
print(f"\n🏁 Manifold procesado y listo en memoria. Filas: {df_final.count():,}")

🚀 Inyectando Inteligencia Semántica al Manifold (1,010,001 filas)...
✅ VOBO TOTAL: Sincronización perfecta. 100% de las zonas tienen nombre.

🧪 MUESTRA ALEATORIA DEL UNIVERSO DENORMALIZADO:
--------------------------------------------------------------------------------
+------------+---------------+-----------------------------------------------------------------------+------------+-----------------+
|product_name|dropoff_zone_id|semantic_name                                                          |upfront_fare|est_trip_time_sec|
+------------+---------------+-----------------------------------------------------------------------+------------+-----------------+
|Mid-Tier    |P_18           |lomas_prado_norte__lomas_trastevere                                    |65.76312    |815.4459         |
|X           |P_17           |lomas_barrilaco__lomas_olimpo__nodo_monte_libano                       |67.50807    |1218.6238        |
|Mid-Tier    |P_21           |fuentes_casino__sedena__tecam

In [14]:
# ==============================================================================
# CELL 6.8: INSTALL GEOSPATIAL TOOLKIT
# ==============================================================================
print("⏳ Instalando librerías Geométricas (Geopandas, Shapely, Pyproj)...")
# Instalación silenciosa de los componentes necesarios
!pip install geopandas shapely pyproj -q

# La instalación en Colab a veces necesita un pequeño hack para el entorno
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "rtree", "-q"])

print("✅ Herramientas Geométricas listas. Iniciando Forja Semántica.")

⏳ Instalando librerías Geométricas (Geopandas, Shapely, Pyproj)...
✅ Herramientas Geométricas listas. Iniciando Forja Semántica.


In [15]:
# ==============================================================================
# PATCH: ASEGURAR H3 Y SHAPELY EN EL ENTORNO
# ==============================================================================
import sys
import subprocess

print("⏳ Reinstalando H3 y Geometría para asegurar la persistencia...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "h3", "geopandas", "shapely", "rtree", "-q"])

# Importación Global para evitar NameError
import h3
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union

print("✅ Todas las librerías Geométricas están ahora disponibles globalmente.")

⏳ Reinstalando H3 y Geometría para asegurar la persistencia...
✅ Todas las librerías Geométricas están ahora disponibles globalmente.


In [17]:
# ==============================================================================
# CELL 6.9: FORJA DE LA DIMENSIÓN DE ORIGEN (H3 to Polygon Name) - STANDALONE
# ==============================================================================
# System: Pienza Semantic Bridge
# Purpose: Map H3 Hexagons to human-readable Polygons using a GeoJSON reference.
# ==============================================================================
import geopandas as gpd
from shapely.geometry import Point
import h3
import os

# --- 1. DEFINICIÓN DE RUTAS (Soberanía de Phase_6) ---
# Inyectamos las rutas aquí para evitar NameError tras reinicios de kernel
MANIFOLD_PATH = "/content/drive/MyDrive/_Pienza/Assets/Phase_4/pienza_manifold_v3.parquet"
GEOJSON_PATH  = "/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly.geojson"
OUTPUT_DICT_PATH = "/content/drive/MyDrive/_Pienza/Assets/Phase_6/dim_pickup_h3_dictionary.parquet"

print("⏳ 1. Leyendo Geometría Soberana (GeoJSON)...")
if not os.path.exists(GEOJSON_PATH):
    raise FileNotFoundError(f"🔴 No se encontró el GeoJSON en: {GEOJSON_PATH}")

gdf_polygons = gpd.read_file(GEOJSON_PATH).to_crs("EPSG:4326")

# --- 2. EXTRACCIÓN DE LLAVES H3 DESDE EL MANIFOLD ---
print("⏳ 2. Extrayendo llaves H3 únicas del Manifold V3...")
if not os.path.exists(MANIFOLD_PATH):
    raise FileNotFoundError(f"🔴 No se encontró el Manifold en: {MANIFOLD_PATH}")

# Leemos el archivo físico usando Spark (que ya debe estar iniciado)
df_h3_keys = spark.read.parquet(MANIFOLD_PATH).select("pickup_h3_id").distinct()
df_h3_keys_pd = df_h3_keys.toPandas()

# --- 3. CONVERSIÓN H3 A COORDENADAS (API v4) ---
print(f"⏳ 3. Procesando {len(df_h3_keys_pd)} hexágonos únicos...")
# Usamos cell_to_latlng para compatibilidad con la versión instalada
df_h3_keys_pd['lat'] = df_h3_keys_pd['pickup_h3_id'].apply(lambda x: h3.cell_to_latlng(x)[0])
df_h3_keys_pd['lon'] = df_h3_keys_pd['pickup_h3_id'].apply(lambda x: h3.cell_to_latlng(x)[1])

# Crear Geopandas DataFrame para los centroides
gdf_centroids = gpd.GeoDataFrame(
    df_h3_keys_pd,
    geometry=gpd.points_from_xy(df_h3_keys_pd.lon, df_h3_keys_pd.lat),
    crs="EPSG:4326"
)

# --- 4. ALGORITMO POINT-IN-POLYGON (Spatial Join) ---
print("⏳ 4. Ejecutando Bautismo Geométrico (Spatial Join)...")
# Cruzamos los puntos con los polígonos
# Renombramos 'name' para evitar conflictos
gdf_joined = gpd.sjoin(
    gdf_centroids,
    gdf_polygons[['name', 'geometry']].rename(columns={'name':'poly_name_temp'}),
    how="left",
    predicate='within'
)

# --- 5. FORMALIZACIÓN DEL DICCIONARIO ---
print("⏳ 5. Resolviendo huérfanos y persistiendo...")
df_dim_pickup_geo = gdf_joined.rename(columns={'poly_name_temp': 'pickup_zone_name'})
df_dim_pickup_geo['pickup_zone_name'] = df_dim_pickup_geo['pickup_zone_name'].fillna("External Logistics Area")

# Limpieza de duplicados y selección final
df_dim_pickup_geo = df_dim_pickup_geo[['pickup_h3_id', 'pickup_zone_name']].drop_duplicates()

# Guardado en Drive
df_dim_pickup_geo.to_parquet(OUTPUT_DICT_PATH, index=False)

print("\n✅ DIMENSIÓN DE ORIGEN FORJADA.")
print(f"   📍 Archivo: {os.path.basename(OUTPUT_DICT_PATH)}")
print(f"   📊 Cobertura: {len(df_dim_pickup_geo)} IDs mapeados.")

display(df_dim_pickup_geo.sort_values(by='pickup_zone_name'))

⏳ 1. Leyendo Geometría Soberana (GeoJSON)...
⏳ 2. Extrayendo llaves H3 únicas del Manifold V3...
⏳ 3. Procesando 32 hexágonos únicos...
⏳ 4. Ejecutando Bautismo Geométrico (Spatial Join)...
⏳ 5. Resolviendo huérfanos y persistiendo...

✅ DIMENSIÓN DE ORIGEN FORJADA.
   📍 Archivo: dim_pickup_h3_dictionary.parquet
   📊 Cobertura: 32 IDs mapeados.


,pickup_h3_id,pickup_zone_name
0,8649864e7ffffff,External Logistics Area
29,864995af7ffffff,External Logistics Area
28,864995bb7ffffff,External Logistics Area
27,864995b8fffffff,External Logistics Area
26,864995b9fffffff,External Logistics Area
25,864995877ffffff,External Logistics Area
24,864995847ffffff,External Logistics Area
23,864995aa7ffffff,External Logistics Area
22,864995b2fffffff,External Logistics Area
21,864995a37ffffff,External Logistics Area


In [20]:
# ==============================================================================
# CELL 6.9-AUDIT: FULL PICKUP ZONE REPORT
# ==============================================================================
# Purpose: Display all 32 unique H3 zones and their assigned semantic names.
# ==============================================================================

print("📖 AUDITORÍA DE DENSIDAD DE ORIGEN (32 ZONAS ÚNICAS):")
print("-" * 80)

# 1. Cargamos el diccionario recién forjado
PICKUP_DICT_PATH = "/content/drive/MyDrive/_Pienza/Assets/Phase_6/dim_pickup_h3_dictionary.parquet"
df_audit = pd.read_parquet(PICKUP_DICT_PATH)

# 2. Re-ordenamos para una inspección lógica (por nombre de zona)
df_audit = df_audit.sort_values(by='pickup_zone_name')

# 3. Mostramos el resultado
pd.set_option('display.max_rows', 100)
display(df_audit)

print("-" * 80)
print(f"✅ Total: {len(df_audit)} zonas de origen listas para el JOIN.")

📖 AUDITORÍA DE DENSIDAD DE ORIGEN (32 ZONAS ÚNICAS):
--------------------------------------------------------------------------------


,pickup_h3_id,pickup_zone_name
0,8649864e7ffffff,External Logistics Area
29,864995af7ffffff,External Logistics Area
28,864995bb7ffffff,External Logistics Area
27,864995b8fffffff,External Logistics Area
26,864995b9fffffff,External Logistics Area
25,864995877ffffff,External Logistics Area
24,864995847ffffff,External Logistics Area
23,864995aa7ffffff,External Logistics Area
22,864995b2fffffff,External Logistics Area
21,864995a37ffffff,External Logistics Area


--------------------------------------------------------------------------------
✅ Total: 32 zonas de origen listas para el JOIN.


In [22]:
# ==============================================================================
# CELL 6.10: DIAGNÓSTICO RÁPIDO (FOLIUM VERSION - STANDALONE)
# ==============================================================================
import folium
import pandas as pd
import geopandas as gpd

print("🛰️  Iniciando Renderizado Forense (Folium)...")

# 1. RECONSTRUCCIÓN DE PUNTOS PARA MAPEO
# Tomamos los centroides calculados en la 6.9
df_h3_points = gdf_centroids.copy()

# Creamos la bandera de "Bautizado" (Si no es 'External Logistics Area')
# Usamos el diccionario df_dim_pickup_geo generado en la celda anterior
bautized_ids = df_dim_pickup_geo[df_dim_pickup_geo['pickup_zone_name'] != "External Logistics Area"]['pickup_h3_id'].tolist()
df_h3_points['is_bautized'] = df_h3_points['pickup_h3_id'].isin(bautized_ids)

# 2. CONFIGURACIÓN DEL MAPA
# Centramos en el corazón de la operación (Anzures/Polanco)
m = folium.Map(location=[19.4326, -99.1332], zoom_start=12, tiles="cartodbpositron")

# 3. CAPA ROJA: TUS POLÍGONOS MASTER
folium.GeoJson(
    gdf_polygons,
    name="Zona Soberana (Polígonos)",
    style_function=lambda x: {
        'fillColor': 'red',
        'color': 'red',
        'weight': 1,
        'fillOpacity': 0.15
    },
    tooltip=folium.GeoJsonTooltip(fields=['name'], aliases=['Zona:'])
).add_to(m)

# 4. CAPA DE PUNTOS: LOS 33 HEXÁGONOS
print(f"📍 Mapeando {len(df_h3_points)} centroides H3...")

for _, row in df_h3_points.iterrows():
    # Verde = Cayó dentro | Gris = Cayó fuera
    color = '#2ecc71' if row['is_bautized'] else '#95a5a6'

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=f"ID: {row['pickup_h3_id']}<br>Bautizado: {row['is_bautized']}"
    ).add_to(m)

# 5. CONTROL DE CAPAS
folium.LayerControl().add_to(m)

print("\n🟢 VERDE: Bautizado (Inside) | ⚪ GRIS: External (Outside) | 🔴 ROJO: Polígono Master")
m

🛰️  Iniciando Renderizado Forense (Folium)...
📍 Mapeando 32 centroides H3...

🟢 VERDE: Bautizado (Inside) | ⚪ GRIS: External (Outside) | 🔴 ROJO: Polígono Master
